# System Ticket Data Summary: Step-by-Step Notebook

Prototype for the **Streamlit App for System Ticket Data Summary** task.

**Pipeline steps:**
1. Load the raw ticket text file into a DataFrame (validate, never repair silently)
2. Repair the 19 shifted rows (missing N/A in the raw file), with a printed report
3. Convert & save as CSV / Excel
4. Clean the data (normalize `N/A`, parse timestamps, drop empty columns)
5. Filter to the predefined service categories `['HDW', 'NET', 'KAI', 'KAV', 'GIGA', 'VOD', 'KAD']`
6. Map categories -> products (Broadband, Voice, TV, GIGA, VOD)
7. Quick EDA of the cleaned data
8. Set up the LLM (Gemini via LangChain) and the 5-section storytelling prompt
9. Assemble everything end-to-end as a LangGraph graph
10. Run the graph -> AI summaries per product
11. Bonus: trends & patterns analysis

The same logic is then ported to `app.py` (Streamlit).

## Step 0: Imports & configuration
Load secrets from `.env` and define the constants used across the pipeline.

In [2]:
import os
import csv
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads GOOGLE_API_KEY and GEMINI_MODEL from .env - never hard-code the key

# The notebook lives in notebooks/; project files live one level up.
# ROOT finds the project folder whether the kernel starts here or at the repo root.
ROOT = Path.cwd() if (Path.cwd() / "Ticket Data (2).txt").exists() else Path.cwd().parent
RAW_FILE = ROOT / "Ticket Data (2).txt"
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

# Only these service categories are kept for analysis (per task description)
ALLOWED_CATEGORIES = ["HDW", "NET", "KAI", "KAV", "GIGA", "VOD", "KAD"]

# Category -> Product mapping (per task description).
# Note: HDW passes the category filter but is not mapped in the doc,
# so we keep it under its own "Hardware" product to avoid silently dropping it.
CATEGORY_TO_PRODUCT = {
    "KAI": "Broadband",
    "NET": "Broadband",
    "KAV": "Voice",
    "KAD": "TV",
    "GIGA": "GIGA",
    "VOD": "VOD",
    "HDW": "Hardware",
}

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
print(f"Model: {GEMINI_MODEL} | API key loaded: {HAS_KEY}")

Model: gemma-4-31b-it | API key loaded: True


## Step 1: Load the raw text file into a DataFrame

The raw `.txt` is a comma-separated table: **line 1** = 38 column names, **each following line** = one ticket.

- `encoding="utf-8-sig"` - strips the **BOM** (`﻿`) that Windows/Excel puts at the start of the file. With plain `utf-8` it stayed glued to the first column name (`"﻿ORDER_NUMBER"`), which caused the `KeyError: 'ORDER_NUMBER'` bug.
- **Validation rule: raise, don't repair.** A row whose field count != 38 is an incomplete ticket - the loader stops and reports the exact line numbers instead of filling with dummy values.

### Explore: parse the file into `rows`

`rows` = a **list of lists**: `rows[0]` is the header (38 column names), every other item is one ticket as a list of 38 string values.

In [3]:
with open(RAW_FILE, "r", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

rows

[['ORDER_NUMBER',
  'ORDER_ID',
  'ORDER_UNIT_ID',
  'ACCEPTANCE_TIME',
  'COMPLETION_TIME',
  'CUSTOMER_NUMBER',
  'CUSTOMER_COUNT',
  'ORDER_TYPE',
  'ORDER_CLASS',
  'PROCESSING_STATUS',
  'SERVICE_CATEGORY',
  'ORDER_DESCRIPTION_1',
  'ORDER_DESCRIPTION_2',
  'ORDER_DESCRIPTION_3_MAXIMUM',
  'ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM',
  'NOTE_MAXIMUM',
  'PLANNING_GROUP_KB',
  'COMPLETION_RESULT_KB',
  'REFERENCE_COMPLETION_RESULT',
  'COMPLETION_NOTE_MAXIMUM',
  'NETWORK_LEVEL',
  'CAUSE',
  'REFERENCE_ERROR_CAUSE',
  'SERVICE_PROVIDER',
  'REFERENCE_SERVICE_PROVIDER',
  'SUBUNIT_NAME',
  'CUSTOMER_COMPLETION_TIME',
  'PROCESSING_END_TIME_MAXIMUM',
  'PROCESSING_END_TIME_MINIMUM',
  'ACCEPTANCE_TIME_MINIMUM',
  'ASSIGNMENT_TIME_MINIMUM',
  'IMIL_TIME_MINIMUM',
  'CUSTOMER_TIME_MINIMUM',
  'START_TIME_MINIMUM',
  'ASSIGNMENT_TIME',
  'ASSIGNED_BY_NAME',
  'ASSIGNMENT_PROCESSING_STATUS',
  'ASSIGNMENT_ADDITIONAL_INFO'],
 ['001-0671177/24',
  '114680574',
  'N/A',
  '11/05/2024 11:00',
 

### Explore: split header from ticket rows

`rows[0]` -> column names; `rows[1:]` -> the 42 data rows.

In [4]:
header,rows = rows[0], rows[1:]
header

['ORDER_NUMBER',
 'ORDER_ID',
 'ORDER_UNIT_ID',
 'ACCEPTANCE_TIME',
 'COMPLETION_TIME',
 'CUSTOMER_NUMBER',
 'CUSTOMER_COUNT',
 'ORDER_TYPE',
 'ORDER_CLASS',
 'PROCESSING_STATUS',
 'SERVICE_CATEGORY',
 'ORDER_DESCRIPTION_1',
 'ORDER_DESCRIPTION_2',
 'ORDER_DESCRIPTION_3_MAXIMUM',
 'ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM',
 'NOTE_MAXIMUM',
 'PLANNING_GROUP_KB',
 'COMPLETION_RESULT_KB',
 'REFERENCE_COMPLETION_RESULT',
 'COMPLETION_NOTE_MAXIMUM',
 'NETWORK_LEVEL',
 'CAUSE',
 'REFERENCE_ERROR_CAUSE',
 'SERVICE_PROVIDER',
 'REFERENCE_SERVICE_PROVIDER',
 'SUBUNIT_NAME',
 'CUSTOMER_COMPLETION_TIME',
 'PROCESSING_END_TIME_MAXIMUM',
 'PROCESSING_END_TIME_MINIMUM',
 'ACCEPTANCE_TIME_MINIMUM',
 'ASSIGNMENT_TIME_MINIMUM',
 'IMIL_TIME_MINIMUM',
 'CUSTOMER_TIME_MINIMUM',
 'START_TIME_MINIMUM',
 'ASSIGNMENT_TIME',
 'ASSIGNED_BY_NAME',
 'ASSIGNMENT_PROCESSING_STATUS',
 'ASSIGNMENT_ADDITIONAL_INFO']

In [5]:
rows

[['001-0671177/24',
  '114680574',
  'N/A',
  '11/05/2024 11:00',
  '11/05/2024 11:30',
  '123',
  '3',
  'Short Ticket',
  'SK',
  'Completed',
  'HDW',
  'Cable Router',
  'WLAN',
  'Slow',
  'N/A',
  'WLAN slow',
  'TSCW2',
  'WLAN settings optimized',
  'N/A',
  'OK',
  'NZ',
  'URS_KIP_Reset_WLAN_Settings',
  'N/A',
  'Customer',
  'Customer',
  'N/A',
  '11/05/2024 11:30',
  '11/05/2024 11:00',
  '11/05/2024 11:00',
  '11/05/2024 11:00',
  'N/A',
  'N/A',
  '11/05/2024 11:00',
  'N/A',
  'N/A',
  'N/A',
  'N/A',
  'N/A'],
 ['001-0682295/24',
  '114691514',
  'N/A',
  '11/05/2024 13:00',
  '11/05/2024 13:45',
  '123',
  '3',
  'Short Ticket',
  'SK',
  'Completed',
  'NTF',
  'Request - KIP',
  'Contract',
  'Installation process/date',
  'N/A',
  'Installation request',
  'TSCKT',
  'Technician appointment scheduled',
  'N/A',
  'OK',
  'NZ',
  'CM_BERA_KD_Complaint_about_BI_Installation',
  'N/A',
  'Customer',
  'Customer',
  'N/A',
  '11/05/2024 13:45',
  '11/05/2024 13:00',
 

### Explore: column count & broken-ticket check

`n_cols` = number of header fields (38). Any ticket row with a different field count is broken/incomplete - here we count them (0 in this file).

In [6]:
### number of cols
n_cols = len(header)
n_cols

38

In [7]:
broken_rows = sum(1 for r in rows if len(r) != len(header))
broken_rows

0

In [8]:
def load_raw_tickets(path):
    """Parse raw tickets; raise if any ticket row is incomplete/malformed."""
    with open(path, "r", encoding="utf-8-sig") as f:   # utf-8-sig strips the BOM
        parsed = list(csv.reader(f))

    header, data = parsed[0], parsed[1:]
    n_cols = len(header)

    # A ticket with the wrong number of fields is corrupt -> report, don't repair
    bad = [(i + 2, len(r)) for i, r in enumerate(data) if r and len(r) != n_cols]  # +2: header is line 1
    if bad:
        raise ValueError(
            f"Incomplete tickets - expected {n_cols} fields per row, "
            f"but got (line, fields): {bad}"
        )

    data = [r for r in data if r]   # ignore blank trailing lines
    print(f"Loaded {len(data)} tickets x {n_cols} columns - all rows complete")
    return header, data

header, rows = load_raw_tickets(RAW_FILE)

Loaded 42 tickets x 38 columns - all rows complete


### Check: what the loader returns

Not a DataFrame yet - on purpose. The rows stay plain Python lists so the next step can repair them with a simple `insert`, which lists support and DataFrame rows don't. The DataFrame gets built right after the repair.

In [9]:
print(len(header), "columns |", len(rows), "tickets")
header[:4], rows[0][:4]

38 columns | 42 tickets


(['ORDER_NUMBER', 'ORDER_ID', 'ORDER_UNIT_ID', 'ACCEPTANCE_TIME'],
 ['001-0671177/24', '114680574', 'N/A', '11/05/2024 11:00'])

## Step 1.5: repair the shifted tickets

19 raw rows (all NET/KAI/KAV/GIGA/VOD/KAD tickets + 1 HDW) were written to the file with one missing `N/A` placeholder: from ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM onward every value sits one column too far left, and an extra `N/A` at the row end keeps the field count at 38 - which is why the broken-ticket check in Step 1 cannot see it.

The repair is two list operations per broken row:
- `insert(add_idx, "N/A")` - put the forgotten placeholder back; everything after it slides one position right automatically
- `pop()` - remove the padding that now hangs beyond the last column

Nothing is invented, and every repaired ticket number is reported. Without this, the LLM would read team codes as customer notes (`note: TSCW2`).

### Explore: find the two positions we need

`header.index(...)` gives a column's position in the row: `add_idx` = where the missing N/A belongs (ADDITIONAL), `note_idx` = where we look for the fingerprint (NOTE).

In [10]:
add_idx = header.index("ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM")
note_idx = header.index("NOTE_MAXIMUM")
add_idx, note_idx

(14, 15)

### Explore: fingerprint the shifted rows

The team codes `TSCW2 / TSCKT / TSCS2` belong only in PLANNING_GROUP_KB. So a row with a team code at the NOTE position must be shifted. The slice preview shows the wrong placement: real note, then team code, then the real fix - each one column too early.

In [11]:
TEAM_CODES = {"TSCW2", "TSCKT", "TSCS2"}

shifted_rows = [r for r in rows if r[note_idx] in TEAM_CODES]
print(f"{len(shifted_rows)} shifted rows found")
shifted_rows[0][add_idx:add_idx + 4]   # [real note, team code, real fix, ...] - all one column early

19 shifted rows found


['Downloadgeschwindigkeit gering', 'TSCKT', 'Bandbreite erhöht', 'N/A']

In [12]:
shifted_rows

[['001-0682299/24',
  '114691518',
  'N/A',
  '11/13/2024 12:00',
  '11/13/2024 12:45',
  '123',
  '3',
  'Kurzticket',
  'SK',
  'ab',
  'NET',
  'Internet',
  'langsam',
  'N/A',
  'Downloadgeschwindigkeit gering',
  'TSCKT',
  'Bandbreite erhöht',
  'N/A',
  'OK',
  'NZ',
  'CM_BERA_KD_Bandbreite',
  'N/A',
  'Kunde',
  'Kunde',
  'N/A',
  '11/13/2024 12:45',
  '11/13/2024 12:00',
  '11/13/2024 12:00',
  '11/13/2024 12:00',
  'N/A',
  'N/A',
  '11/13/2024 12:00',
  'N/A',
  'N/A',
  'N/A',
  'N/A',
  'N/A',
  'N/A'],
 ['001-0670955/24',
  '114680355',
  'N/A',
  '11/14/2024 08:00',
  '11/14/2024 08:30',
  '123',
  '3',
  'Kurzticket',
  'SK',
  'ab',
  'KAI',
  'CI-Modul',
  'kein Empfang',
  'N/A',
  'CI-Modul defekt',
  'TSCW2',
  'Neues CI-Modul bestellt',
  'N/A',
  'OK',
  'NZ',
  'URS_KIP_CI_Modul_Defekt',
  'N/A',
  'Kunde',
  'Kunde',
  'N/A',
  '11/14/2024 08:30',
  '11/14/2024 08:00',
  '11/14/2024 08:00',
  '11/14/2024 08:00',
  'N/A',
  'N/A',
  '11/14/2024 08:00',
  'N/

### Explore: watch insert + pop on one copied row

`insert` puts N/A at `add_idx` and slides everything after it one right (39 fields for a moment); `pop` removes the last item - the padding that now sits beyond the last column - bringing the row back to 38. The removed item is shown so you can see it was junk.

In [13]:
demo = shifted_rows[0].copy()      # work on a copy so the real row stays untouched here
demo.insert(add_idx, "N/A")        # everything after add_idx slides one position right
junk = demo.pop()                  # remove the padding beyond the last column
print("removed item:", repr(junk), "| fields:", len(demo))
demo[add_idx:add_idx + 4]          # [N/A, real note, team code, real fix] - aligned now

removed item: 'N/A' | fields: 38


['N/A', 'Downloadgeschwindigkeit gering', 'TSCKT', 'Bandbreite erhöht']

### The repair function: insert, pop, report - then build the DataFrame

The two list operations from above, applied only to fingerprinted rows. Every repaired ticket number is printed. With the rows aligned, we can finally build the DataFrame safely.

In [14]:
def repair_shifted_rows(rows: list, header: list) -> list:
    """Fix rows written with one missing N/A: insert it back, everything slides right."""
    add_idx = header.index("ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM")
    note_idx = header.index("NOTE_MAXIMUM")

    repaired = []
    for r in rows:
        if r[note_idx] in TEAM_CODES:   # team code where the note belongs = shifted row
            r.insert(add_idx, "N/A")    # insert the forgotten N/A -> rest slides right
            r.pop()                     # drop the padding beyond the last column
            repaired.append(r[0])       # remember its order number

    print(f"Repaired {len(repaired)} shifted tickets: {repaired}")
    return rows

rows = repair_shifted_rows(rows, header)
df = pd.DataFrame(rows, columns=header)
df.head()

Repaired 19 shifted tickets: ['001-0682299/24', '001-0670955/24', '001-0671372/24', '001-0670932/24', '001-0670744/24', '001-0671183/24', '001-0670956/24', '001-0671373/24', '001-0670933/24', '001-0670745/24', '001-0671184/24', '001-0682301/24', '001-0671185/24', '001-0682302/24', '001-0670957/24', '001-0671374/24', '001-0670934/24', '001-0670746/24', '001-0671186/24']


,ORDER_NUMBER,ORDER_ID,ORDER_UNIT_ID,ACCEPTANCE_TIME,COMPLETION_TIME,CUSTOMER_NUMBER,CUSTOMER_COUNT,ORDER_TYPE,ORDER_CLASS,PROCESSING_STATUS,...,PROCESSING_END_TIME_MINIMUM,ACCEPTANCE_TIME_MINIMUM,ASSIGNMENT_TIME_MINIMUM,IMIL_TIME_MINIMUM,CUSTOMER_TIME_MINIMUM,START_TIME_MINIMUM,ASSIGNMENT_TIME,ASSIGNED_BY_NAME,ASSIGNMENT_PROCESSING_STATUS,ASSIGNMENT_ADDITIONAL_INFO
0,001-0671177/24,114680574,N/A,11/05/2024 11:00,11/05/2024 11:30,123,3,Short Ticket,SK,Completed,...,11/05/2024 11:00,11/05/2024 11:00,N/A,N/A,11/05/2024 11:00,N/A,N/A,N/A,N/A,N/A
1,001-0682295/24,114691514,N/A,11/05/2024 13:00,11/05/2024 13:45,123,3,Short Ticket,SK,Completed,...,11/05/2024 13:00,11/05/2024 13:00,N/A,N/A,11/05/2024 13:00,N/A,N/A,N/A,N/A,N/A
2,001-0670952/24,114680352,N/A,11/06/2024 08:00,11/06/2024 08:30,123,3,Short Ticket,SK,Completed,...,11/06/2024 08:00,11/06/2024 08:00,N/A,N/A,11/06/2024 08:00,N/A,N/A,N/A,N/A,N/A
3,001-0671369/24,114680756,N/A,11/06/2024 10:00,11/06/2024 10:15,123,3,Short Ticket,SK,Completed,...,11/06/2024 10:00,11/06/2024 10:00,N/A,N/A,11/06/2024 10:00,N/A,N/A,N/A,N/A,N/A
4,001-0670929/24,114680329,N/A,11/06/2024 12:00,11/06/2024 13:00,123,3,Task,SK,Completed,...,11/06/2024 12:00,11/06/2024 12:00,N/A,N/A,11/06/2024 12:00,N/A,N/A,N/A,N/A,N/A


### Check: values back under the right headers

The previously shifted tickets, now as DataFrame rows: the note text is in NOTE_MAXIMUM, the team code back in PLANNING_GROUP_KB, the fix in COMPLETION_RESULT_KB.

In [15]:
df.loc[df["ORDER_NUMBER"].isin([r[0] for r in shifted_rows]),
       ["ORDER_NUMBER", "ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM",
        "NOTE_MAXIMUM", "PLANNING_GROUP_KB", "COMPLETION_RESULT_KB"]].head()

,ORDER_NUMBER,ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM,NOTE_MAXIMUM,PLANNING_GROUP_KB,COMPLETION_RESULT_KB
22,001-0682299/24,N/A,Downloadgeschwindigkeit gering,TSCKT,Bandbreite erhöht
23,001-0670955/24,N/A,CI-Modul defekt,TSCW2,Neues CI-Modul bestellt
24,001-0671372/24,N/A,Bestimmte Sender fehlen,TSCW2,Sendeanlage geprüft
25,001-0670932/24,N/A,Gigabit Option nicht aktivierbar,TSCS2,SIT12
26,001-0670744/24,N/A,Filme laden nicht,TSCS2,TITAN erfolgreich


## Step 2: Convert to CSV / Excel
Persist the converted data so downstream tools (and the Streamlit app) can reuse it.

In [16]:
csv_path = DATA_DIR / "tickets_converted.csv"
xlsx_path = DATA_DIR / "tickets_converted.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)
print(f"Saved: {csv_path} ({csv_path.stat().st_size:,} bytes)")
print(f"Saved: {xlsx_path} ({xlsx_path.stat().st_size:,} bytes)")

Saved: d:\VOIS\data\tickets_converted.csv (16,133 bytes)
Saved: d:\VOIS\data\tickets_converted.xlsx (13,660 bytes)


## Step 3: Data cleaning

What `clean_tickets` does, line by line:

- `df.copy()` - never mutate the input; each step returns a **new** DataFrame.
- `.apply(lambda s: s.str.strip())` - trim stray spaces in every column.
- `.replace({"N/A": pd.NA, "": pd.NA})` - turn the literal text `N/A` / empty strings into **real** missing values, so `pd.notna()` checks work later.
- two explicit `pd.to_datetime(..., format="%m/%d/%Y %H:%M")` calls - parse the timestamps. Default `errors="raise"`: a garbage date **stops the pipeline loudly** (raise, don't repair).
- `RESOLUTION_MINUTES` = completion − acceptance, in minutes - how fast each ticket was solved.
- finally, **drop columns that are entirely empty** and print which ones were dropped (report, never silent).

### Explore: the table before cleaning

All text, literal `N/A` strings everywhere, dates still strings.

In [17]:
df

,ORDER_NUMBER,ORDER_ID,ORDER_UNIT_ID,ACCEPTANCE_TIME,COMPLETION_TIME,CUSTOMER_NUMBER,CUSTOMER_COUNT,ORDER_TYPE,ORDER_CLASS,PROCESSING_STATUS,...,PROCESSING_END_TIME_MINIMUM,ACCEPTANCE_TIME_MINIMUM,ASSIGNMENT_TIME_MINIMUM,IMIL_TIME_MINIMUM,CUSTOMER_TIME_MINIMUM,START_TIME_MINIMUM,ASSIGNMENT_TIME,ASSIGNED_BY_NAME,ASSIGNMENT_PROCESSING_STATUS,ASSIGNMENT_ADDITIONAL_INFO
0,001-0671177/24,114680574,N/A,11/05/2024 11:00,11/05/2024 11:30,123,3,Short Ticket,SK,Completed,...,11/05/2024 11:00,11/05/2024 11:00,N/A,N/A,11/05/2024 11:00,N/A,N/A,N/A,N/A,N/A
1,001-0682295/24,114691514,N/A,11/05/2024 13:00,11/05/2024 13:45,123,3,Short Ticket,SK,Completed,...,11/05/2024 13:00,11/05/2024 13:00,N/A,N/A,11/05/2024 13:00,N/A,N/A,N/A,N/A,N/A
2,001-0670952/24,114680352,N/A,11/06/2024 08:00,11/06/2024 08:30,123,3,Short Ticket,SK,Completed,...,11/06/2024 08:00,11/06/2024 08:00,N/A,N/A,11/06/2024 08:00,N/A,N/A,N/A,N/A,N/A
3,001-0671369/24,114680756,N/A,11/06/2024 10:00,11/06/2024 10:15,123,3,Short Ticket,SK,Completed,...,11/06/2024 10:00,11/06/2024 10:00,N/A,N/A,11/06/2024 10:00,N/A,N/A,N/A,N/A,N/A
4,001-0670929/24,114680329,N/A,11/06/2024 12:00,11/06/2024 13:00,123,3,Task,SK,Completed,...,11/06/2024 12:00,11/06/2024 12:00,N/A,N/A,11/06/2024 12:00,N/A,N/A,N/A,N/A,N/A
5,001-0670741/24,114680146,N/A,11/07/2024 09:00,11/07/2024 09:15,123,3,Short Ticket,SK,Completed,...,11/07/2024 09:00,11/07/2024 09:00,N/A,N/A,11/07/2024 09:00,N/A,N/A,N/A,N/A,N/A
6,001-0671178/24,114680575,N/A,11/07/2024 11:00,11/07/2024 11:30,123,3,Short Ticket,SK,Completed,...,11/07/2024 11:00,11/07/2024 11:00,N/A,N/A,11/07/2024 11:00,N/A,N/A,N/A,N/A,N/A
7,001-0682296/24,114691515,N/A,11/08/2024 08:00,11/08/2024 08:45,456,4,Short Ticket,SK,Completed,...,11/08/2024 08:00,11/08/2024 08:00,N/A,N/A,11/08/2024 08:00,N/A,N/A,N/A,N/A,N/A
8,001-0670953/24,114680353,N/A,11/08/2024 10:00,11/08/2024 10:30,456,4,Short Ticket,SK,Completed,...,11/08/2024 10:00,11/08/2024 10:00,N/A,N/A,11/08/2024 10:00,N/A,N/A,N/A,N/A,N/A
9,001-0671370/24,114680757,N/A,11/08/2024 12:00,11/08/2024 12:15,456,4,Short Ticket,SK,Completed,...,11/08/2024 12:00,11/08/2024 12:00,N/A,N/A,11/08/2024 12:00,N/A,N/A,N/A,N/A,N/A


### Explore: copy, strip, normalize N/A

Three lines on a scratch copy `tmp` (the real `df` is only transformed by the final function). After `replace`, the literal `N/A` strings show up as `NaN` = real missing values that `pd.notna()` understands.

In [18]:
tmp = df.copy()                                  # work on a copy, never mutate the input
tmp = tmp.apply(lambda s: s.str.strip())         # remove stray spaces, column by column
tmp = tmp.replace({"N/A": pd.NA, "": pd.NA})     # literal 'N/A' / '' -> real missing values
tmp[["ORDER_DESCRIPTION_3_MAXIMUM", "CAUSE"]].head(10)

,ORDER_DESCRIPTION_3_MAXIMUM,CAUSE
0,Slow,URS_KIP_Reset_WLAN_Settings
1,Installation process/date,CM_BERA_KD_Complaint_about_BI_Installation
2,No connection,URS_KIP_No_Connection
3,Unstable,BM_STOE_HDW_Malfunction_KIP_WLAN_Without_Funct...
4,Preliminary test result see note field,SIT11
5,Preliminary test result see note field,k03
6,No connection,URS_KIP_Reset_WLAN_Settings
7,Invoice Clarification,CM_BERA_KD_Invoice
8,Slow,URS_KIP_Slow_Connection
9,Stable,BM_STOE_HDW_FunktSt_KIP_WLAN_ohne_Funkt_nicht


### Explore: parse the timestamps

Text like `11/05/2024 11:00` becomes a real datetime. `format="%m/%d/%Y %H:%M"` tells pandas the exact layout; the default `errors="raise"` means a garbage date stops the run loudly (raise, don't repair).

In [19]:
tmp["ACCEPTANCE_TIME"] = pd.to_datetime(tmp["ACCEPTANCE_TIME"], format="%m/%d/%Y %H:%M")
tmp["COMPLETION_TIME"] = pd.to_datetime(tmp["COMPLETION_TIME"], format="%m/%d/%Y %H:%M")
tmp[["ACCEPTANCE_TIME", "COMPLETION_TIME"]].dtypes

ACCEPTANCE_TIME    datetime64[us]
COMPLETION_TIME    datetime64[us]
dtype: object

### Explore: resolution time in minutes

Subtracting two datetime columns gives a Timedelta per row; `.dt.total_seconds() / 60` converts it to minutes. `describe()` sanity-checks the result (all tickets resolve in 15 to 60 minutes).

In [20]:
tmp["RESOLUTION_MINUTES"] = (tmp["COMPLETION_TIME"] - tmp["ACCEPTANCE_TIME"]).dt.total_seconds() / 60
tmp["RESOLUTION_MINUTES"].describe()

count    42.000000
mean     32.857143
std      15.267647
min      15.000000
25%      15.000000
50%      30.000000
75%      45.000000
max      60.000000
Name: RESOLUTION_MINUTES, dtype: float64

### Explore: find the all-empty columns

`isna().all()` is True only for columns where every one of the 42 values is missing. A column with even a single real value stays.

In [21]:
empty_cols = tmp.columns[tmp.isna().all()]
empty_cols

Index(['ORDER_UNIT_ID', 'ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM', 'SUBUNIT_NAME',
       'ASSIGNMENT_TIME_MINIMUM', 'IMIL_TIME_MINIMUM', 'START_TIME_MINIMUM',
       'ASSIGNMENT_TIME', 'ASSIGNED_BY_NAME', 'ASSIGNMENT_PROCESSING_STATUS',
       'ASSIGNMENT_ADDITIONAL_INFO'],
      dtype='str')

### The cleaning function: all lines together

In [22]:
def clean_tickets(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize missing values, parse timestamps, drop all-empty columns."""
    out = df.copy()
    out = out.apply(lambda s: s.str.strip())
    out = out.replace({"N/A": pd.NA, "": pd.NA})

    out["ACCEPTANCE_TIME"] = pd.to_datetime(out["ACCEPTANCE_TIME"], format="%m/%d/%Y %H:%M")
    out["COMPLETION_TIME"] = pd.to_datetime(out["COMPLETION_TIME"], format="%m/%d/%Y %H:%M")

    out["RESOLUTION_MINUTES"] = (
        (out["COMPLETION_TIME"] - out["ACCEPTANCE_TIME"]).dt.total_seconds() / 60
    )

    empty_cols = out.columns[out.isna().all()]
    if len(empty_cols):
        out = out.drop(columns=empty_cols)
        print(f"Dropped {len(empty_cols)} all-empty columns: {list(empty_cols)}")
    return out

df_clean = clean_tickets(df)
print(df_clean[["ACCEPTANCE_TIME", "COMPLETION_TIME", "RESOLUTION_MINUTES"]].describe())
df_clean.head(3)

Dropped 10 all-empty columns: ['ORDER_UNIT_ID', 'ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM', 'SUBUNIT_NAME', 'ASSIGNMENT_TIME_MINIMUM', 'IMIL_TIME_MINIMUM', 'START_TIME_MINIMUM', 'ASSIGNMENT_TIME', 'ASSIGNED_BY_NAME', 'ASSIGNMENT_PROCESSING_STATUS', 'ASSIGNMENT_ADDITIONAL_INFO']
                  ACCEPTANCE_TIME      COMPLETION_TIME  RESOLUTION_MINUTES
count                          42                   42           42.000000
mean   2024-11-12 07:07:08.571428  2024-11-12 07:40:00           32.857143
min           2024-11-05 11:00:00  2024-11-05 11:30:00           15.000000
25%           2024-11-08 18:45:00  2024-11-08 19:33:45           15.000000
50%           2024-11-12 09:30:00  2024-11-12 09:52:30           30.000000
75%           2024-11-15 02:45:00  2024-11-15 03:26:15           45.000000
max           2024-11-20 11:00:00  2024-11-20 11:30:00           60.000000
std                           NaN                  NaN           15.267647


,ORDER_NUMBER,ORDER_ID,ACCEPTANCE_TIME,COMPLETION_TIME,CUSTOMER_NUMBER,CUSTOMER_COUNT,ORDER_TYPE,ORDER_CLASS,PROCESSING_STATUS,SERVICE_CATEGORY,...,CAUSE,REFERENCE_ERROR_CAUSE,SERVICE_PROVIDER,REFERENCE_SERVICE_PROVIDER,CUSTOMER_COMPLETION_TIME,PROCESSING_END_TIME_MAXIMUM,PROCESSING_END_TIME_MINIMUM,ACCEPTANCE_TIME_MINIMUM,CUSTOMER_TIME_MINIMUM,RESOLUTION_MINUTES
0,001-0671177/24,114680574,2024-11-05 11:00:00,2024-11-05 11:30:00,123,3,Short Ticket,SK,Completed,HDW,...,URS_KIP_Reset_WLAN_Settings,NaN,Customer,Customer,11/05/2024 11:30,11/05/2024 11:00,11/05/2024 11:00,11/05/2024 11:00,11/05/2024 11:00,30.0
1,001-0682295/24,114691514,2024-11-05 13:00:00,2024-11-05 13:45:00,123,3,Short Ticket,SK,Completed,NTF,...,CM_BERA_KD_Complaint_about_BI_Installation,NaN,Customer,Customer,11/05/2024 13:45,11/05/2024 13:00,11/05/2024 13:00,11/05/2024 13:00,11/05/2024 13:00,45.0
2,001-0670952/24,114680352,2024-11-06 08:00:00,2024-11-06 08:30:00,123,3,Short Ticket,SK,Completed,HDW,...,URS_KIP_No_Connection,NaN,Customer,Customer,11/06/2024 08:30,11/06/2024 08:00,11/06/2024 08:00,11/06/2024 08:00,11/06/2024 08:00,30.0


### Check: after cleaning

Timestamps are `datetime64`, `RESOLUTION_MINUTES` is numeric, `N/A` became real missing values (see the Non-Null counts drop).

In [23]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ORDER_NUMBER                 42 non-null     str           
 1   ORDER_ID                     42 non-null     str           
 2   ACCEPTANCE_TIME              42 non-null     datetime64[us]
 3   COMPLETION_TIME              42 non-null     datetime64[us]
 4   CUSTOMER_NUMBER              42 non-null     str           
 5   CUSTOMER_COUNT               42 non-null     str           
 6   ORDER_TYPE                   42 non-null     str           
 7   ORDER_CLASS                  42 non-null     str           
 8   PROCESSING_STATUS            42 non-null     str           
 9   SERVICE_CATEGORY             42 non-null     str           
 10  ORDER_DESCRIPTION_1          42 non-null     str           
 11  ORDER_DESCRIPTION_2          42 non-null     str          

## Step 3.2: how the empty-column drop works

`isna().all()` marks columns where **every** value is missing. (This logic now also runs inside `clean_tickets`, so on a fresh run these cells find nothing left to drop - kept to show the mechanics.)

In [24]:
empty_cols = df_clean.columns[df_clean.isna().all()]
empty_cols



Index([], dtype='str')

## Step 4: Filter to the predefined categories

Only `HDW, NET, KAI, KAV, GIGA, VOD, KAD` are kept - everything else
(`NTF` inquiries, `SOP` activations, ...) is excluded from the analysis.

### Explore: category counts and the `isin` mask

First see which categories exist and their volumes, then preview the filter: `isin(ALLOWED_CATEGORIES)` builds a True/False mask per row, and `df[mask]` keeps only the True rows (SOP and NTF drop out).

In [25]:
df_clean["SERVICE_CATEGORY"].value_counts()

SERVICE_CATEGORY
HDW     14
SOP      6
NTF      4
NET      3
KAI      3
KAV      3
GIGA     3
VOD      3
KAD      3
Name: count, dtype: int64

In [26]:
df_clean[df_clean["SERVICE_CATEGORY"].isin(ALLOWED_CATEGORIES)]

,ORDER_NUMBER,ORDER_ID,ACCEPTANCE_TIME,COMPLETION_TIME,CUSTOMER_NUMBER,CUSTOMER_COUNT,ORDER_TYPE,ORDER_CLASS,PROCESSING_STATUS,SERVICE_CATEGORY,...,CAUSE,REFERENCE_ERROR_CAUSE,SERVICE_PROVIDER,REFERENCE_SERVICE_PROVIDER,CUSTOMER_COMPLETION_TIME,PROCESSING_END_TIME_MAXIMUM,PROCESSING_END_TIME_MINIMUM,ACCEPTANCE_TIME_MINIMUM,CUSTOMER_TIME_MINIMUM,RESOLUTION_MINUTES
0,001-0671177/24,114680574,2024-11-05 11:00:00,2024-11-05 11:30:00,123,3,Short Ticket,SK,Completed,HDW,...,URS_KIP_Reset_WLAN_Settings,NaN,Customer,Customer,11/05/2024 11:30,11/05/2024 11:00,11/05/2024 11:00,11/05/2024 11:00,11/05/2024 11:00,30.0
2,001-0670952/24,114680352,2024-11-06 08:00:00,2024-11-06 08:30:00,123,3,Short Ticket,SK,Completed,HDW,...,URS_KIP_No_Connection,NaN,Customer,Customer,11/06/2024 08:30,11/06/2024 08:00,11/06/2024 08:00,11/06/2024 08:00,11/06/2024 08:00,30.0
3,001-0671369/24,114680756,2024-11-06 10:00:00,2024-11-06 10:15:00,123,3,Short Ticket,SK,Completed,HDW,...,BM_STOE_HDW_Malfunction_KIP_WLAN_Without_Funct...,NaN,Customer,Customer,11/06/2024 10:15,11/06/2024 10:00,11/06/2024 10:00,11/06/2024 10:00,11/06/2024 10:00,15.0
6,001-0671178/24,114680575,2024-11-07 11:00:00,2024-11-07 11:30:00,123,3,Short Ticket,SK,Completed,HDW,...,URS_KIP_Reset_WLAN_Settings,NaN,Customer,Customer,11/07/2024 11:30,11/07/2024 11:00,11/07/2024 11:00,11/07/2024 11:00,11/07/2024 11:00,30.0
8,001-0670953/24,114680353,2024-11-08 10:00:00,2024-11-08 10:30:00,456,4,Short Ticket,SK,Completed,HDW,...,URS_KIP_Slow_Connection,NaN,Customer,Customer,11/08/2024 10:30,11/08/2024 10:00,11/08/2024 10:00,11/08/2024 10:00,11/08/2024 10:00,30.0
9,001-0671370/24,114680757,2024-11-08 12:00:00,2024-11-08 12:15:00,456,4,Short Ticket,SK,Completed,HDW,...,BM_STOE_HDW_FunktSt_KIP_WLAN_ohne_Funkt_nicht,NaN,Customer,Customer,11/08/2024 12:15,11/08/2024 12:00,11/08/2024 12:00,11/08/2024 12:00,11/08/2024 12:00,15.0
12,001-0671179/24,114680576,2024-11-09 11:00:00,2024-11-09 11:30:00,456,4,Short Ticket,SK,Completed,HDW,...,URS_KIP_WLAN-Settings_Reset,NaN,Customer,Customer,11/09/2024 11:30,11/09/2024 11:00,11/09/2024 11:00,11/09/2024 11:00,11/09/2024 11:00,30.0
14,001-0671180/24,114680577,2024-11-10 11:00:00,2024-11-10 11:30:00,789,2,Short Ticket,SK,Completed,HDW,...,URS_KIP_WLAN-Settings_Reset,NaN,Customer,Customer,11/10/2024 11:30,11/10/2024 11:00,11/10/2024 11:00,11/10/2024 11:00,11/10/2024 11:00,30.0
16,001-0670954/24,114680354,2024-11-11 08:00:00,2024-11-11 08:30:00,789,2,Short Ticket,SK,Completed,HDW,...,URS_KIP_No_Connection,NaN,Customer,Customer,11/11/2024 08:30,11/11/2024 08:00,11/11/2024 08:00,11/11/2024 08:00,11/11/2024 08:00,30.0
17,001-0671371/24,114680758,2024-11-11 10:00:00,2024-11-11 10:15:00,789,2,Short Ticket,SK,Completed,HDW,...,BM_STOE_HDW_FunktSt_KIP_WLAN_ohne_Funkt_nicht,NaN,Customer,Customer,11/11/2024 10:15,11/11/2024 10:00,11/11/2024 10:00,11/11/2024 10:00,11/11/2024 10:00,15.0


In [27]:
def filter_categories(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only tickets whose SERVICE_CATEGORY is in the allowed list."""
    before = df["SERVICE_CATEGORY"].value_counts()
    out = df[df["SERVICE_CATEGORY"].isin(ALLOWED_CATEGORIES)].copy()
    out = out.sort_values("ACCEPTANCE_TIME").reset_index(drop=True)
    print("Category counts BEFORE filtering:")
    print(before.to_string())
    print(f"\nKept {len(out)} / {len(df)} tickets")
    return out

df_filtered = filter_categories(df_clean)
df_filtered["SERVICE_CATEGORY"].value_counts()

Category counts BEFORE filtering:
SERVICE_CATEGORY
HDW     14
SOP      6
NTF      4
NET      3
KAI      3
KAV      3
GIGA     3
VOD      3
KAD      3

Kept 32 / 42 tickets


SERVICE_CATEGORY
HDW     14
NET      3
KAI      3
KAV      3
GIGA     3
VOD      3
KAD      3
Name: count, dtype: int64

## Step 5: Map categories to products

| Product | Categories |
|---|---|
| Broadband | KAI, NET |
| Voice | KAV |
| TV | KAD |
| GIGA | GIGA |
| VOD | VOD |
| Hardware* | HDW |

\* HDW survives the category filter but has no product in the task doc - we keep it as its own group.

In [28]:
def map_products(df: pd.DataFrame) -> pd.DataFrame:
    """Add a PRODUCT column derived from SERVICE_CATEGORY."""
    out = df.copy()
    out["PRODUCT"] = out["SERVICE_CATEGORY"].map(CATEGORY_TO_PRODUCT)
    return out

df_products = map_products(df_filtered)
df_products.groupby("PRODUCT")["SERVICE_CATEGORY"].value_counts()

PRODUCT    SERVICE_CATEGORY
Broadband  NET                  3
           KAI                  3
GIGA       GIGA                 3
Hardware   HDW                 14
TV         KAD                  3
VOD        VOD                  3
Voice      KAV                  3
Name: count, dtype: int64

## Step 6: Quick EDA
A sanity look at the cleaned dataset before summarization.

In [29]:
print("Tickets per customer:")
print(df_products["CUSTOMER_NUMBER"].value_counts().to_string())
print("\nTickets per order type:")
print(df_products["ORDER_TYPE"].value_counts().to_string())
print("\nDate range:", df_products["ACCEPTANCE_TIME"].min(), "->", df_products["ACCEPTANCE_TIME"].max())
print("\nAvg resolution minutes per product:")
print(df_products.groupby("PRODUCT")["RESOLUTION_MINUTES"].mean().round(1).to_string())

Tickets per customer:
CUSTOMER_NUMBER
123    11
789    11
456    10

Tickets per order type:
ORDER_TYPE
Short Ticket    23
Kurzticket       6
Task             2
Aufgabe          1

Date range: 2024-11-05 11:00:00 -> 2024-11-20 11:00:00

Avg resolution minutes per product:
PRODUCT
Broadband    32.5
GIGA         45.0
Hardware     27.9
TV           35.0
VOD          20.0
Voice        30.0


## Step 7: LLM setup (Gemini via LangChain)

`gemini-3.5-flash` through `langchain-google-genai`. The key comes from `.env`
(`GOOGLE_API_KEY`) - never hard-coded.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

if not HAS_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY is empty in .env - paste your key "
        "(https://aistudio.google.com/apikey) and re-run from this cell."
    )

# timeout: a stalled connection must raise (so retries can act), never hang forever
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.3, timeout=90)
# .text flattens the model reply (newer Gemini returns a list of content blocks)
print(llm.invoke("Reply with exactly: Gemini connection OK").text)

## Step 8: The 5-section storytelling prompt

Per the task doc, each customer x product summary must contain **Initial Issue,
Follow-ups, Developments, Later Incidents, Recent Events** - each with a
Timeframe, Ticket Numbers, and a Narrative.

In [31]:
for r, y in df_clean.iterrows():
    print(y)

ORDER_NUMBER                                001-0671177/24
ORDER_ID                                         114680574
ACCEPTANCE_TIME                        2024-11-05 11:00:00
COMPLETION_TIME                        2024-11-05 11:30:00
CUSTOMER_NUMBER                                        123
CUSTOMER_COUNT                                           3
ORDER_TYPE                                    Short Ticket
ORDER_CLASS                                             SK
PROCESSING_STATUS                                Completed
SERVICE_CATEGORY                                       HDW
ORDER_DESCRIPTION_1                           Cable Router
ORDER_DESCRIPTION_2                                   WLAN
ORDER_DESCRIPTION_3_MAXIMUM                           Slow
NOTE_MAXIMUM                                     WLAN slow
PLANNING_GROUP_KB                                    TSCW2
COMPLETION_RESULT_KB               WLAN settings optimized
REFERENCE_COMPLETION_RESULT                            N

In [32]:
from langchain_core.prompts import ChatPromptTemplate

STORY_PROMPT = ChatPromptTemplate.from_template("""\
You are a telecom customer-service analyst. Summarize the support-ticket history
of customer **{customer}** for the product **{product}** (service categories:
{categories}) into the five-phase storytelling structure: initial_issue,
follow_ups, developments, later_incidents, recent_events.

Rules:
- Use ONLY the ticket data below; never invent facts or ticket numbers.
- Keep events chronological across the phases.
- Every ticket below must appear in exactly one phase's ticket_numbers.
- Phases with no tickets: ticket_numbers = [] and a one-line narrative such as
  "No further tickets recorded in this period."
- Some tickets are in German - write every narrative in English.

Tickets (chronological):
{tickets}
""")


def format_tickets(df: pd.DataFrame) -> str:
    """Render tickets as compact chronological lines for the LLM."""
    lines = []
    for _, r in df.iterrows():
        parts = [
            str(r["ORDER_NUMBER"]),
            r["ACCEPTANCE_TIME"].strftime("%Y-%m-%d %H:%M") if pd.notna(r["ACCEPTANCE_TIME"]) else "?",
            f"cat={r['SERVICE_CATEGORY']}",
            "issue: " + " / ".join(str(r[c]) for c in [
                "ORDER_DESCRIPTION_1", "ORDER_DESCRIPTION_2", "ORDER_DESCRIPTION_3_MAXIMUM"
            ] if c in r.index and pd.notna(r[c])),
        ]
        if pd.notna(r["NOTE_MAXIMUM"]):
            parts.append(f"note: {r['NOTE_MAXIMUM']}")
        resolution = " / ".join(str(r[c]) for c in [
            "COMPLETION_RESULT_KB", "COMPLETION_NOTE_MAXIMUM"
        ] if c in r.index and pd.notna(r[c]))
        if resolution:
            parts.append(f"resolution: {resolution}")
        lines.append("- " + " | ".join(parts))
    return "\n".join(lines)

print(format_tickets(df_products.head(2)))  # preview of what the LLM receives

- 001-0671177/24 | 2024-11-05 11:00 | cat=HDW | issue: Cable Router / WLAN / Slow | note: WLAN slow | resolution: WLAN settings optimized / OK
- 001-0670952/24 | 2024-11-06 08:00 | cat=HDW | issue: Cable Router / Internet / No connection | note: No internet connection | resolution: Router restarted / OK


### Preview: the exact text block Gemini receives

One compact line per ticket, chronological. After the Step 1.5 repair, every line carries the real note and fix (e.g. `note: Upload Too Slow | resolution: Line Checked / OK`) instead of misplaced team codes.

In [33]:
print(format_tickets(df_products))

- 001-0671177/24 | 2024-11-05 11:00 | cat=HDW | issue: Cable Router / WLAN / Slow | note: WLAN slow | resolution: WLAN settings optimized / OK
- 001-0670952/24 | 2024-11-06 08:00 | cat=HDW | issue: Cable Router / Internet / No connection | note: No internet connection | resolution: Router restarted / OK
- 001-0671369/24 | 2024-11-06 10:00 | cat=HDW | issue: Cable Router / WLAN / Unstable | note: WLAN unstable | resolution: Channel changed / OK
- 001-0671178/24 | 2024-11-07 11:00 | cat=HDW | issue: Cable Router / WLAN / No connection | note: WLAN no connection | resolution: Router reset / OK
- 001-0670953/24 | 2024-11-08 10:00 | cat=HDW | issue: Cable Router / Internet / Slow | note: Internet Slow | resolution: Firmware Update Performed / OK
- 001-0671370/24 | 2024-11-08 12:00 | cat=HDW | issue: Cable Router / WLAN / Stable | note: WLAN Stable | resolution: Channel Change Automatic / OK
- 001-0671179/24 | 2024-11-09 11:00 | cat=HDW | issue: Cable Router / WLAN / No Connection | note: WL

## Step 8.5: the output contract - schema, validation, evaluation

The prose approach trusted the model to produce five sections and real ticket
numbers. This layer stops trusting and starts checking:

1. **Schema** (Pydantic + native structured output): the API itself forces the
   JSON shape - five phases, ticket_numbers as a list of strings.
2. **Evaluation** (the two checklists): every cited ticket must exist in the
   group, every real ticket must be cited. Violations trigger one corrective
   retry that names the exact mistakes.
3. **Computed timeframes**: the model has NO timeframe field - displayed dates
   are looked up from the data after the cited tickets are verified, so a
   displayed date cannot be hallucinated.

Rule of the layer: validators are inspectors, not editors - they reject bad
answers, they never fix them; the model corrects its own work.

### The contract: what the model is allowed to author

Only two fields per phase - ticket references (checkable) and narrative
(grounded). No timeframe: whatever the model does not author, it cannot
hallucinate. The `field_validator` rejects any list item that is not a real
order-number pattern - so `"N/A"` can never enter the data; absence is an
empty list, never a value.

In [34]:
from pydantic import BaseModel, field_validator

import re
ORDER_NUMBER_PATTERN = re.compile(r"\d{3}-\d+/\d{2}")


class Section(BaseModel):
    ticket_numbers: list[str] = []      # [] = this phase covers no tickets
    narrative: str

    @field_validator("ticket_numbers")
    @classmethod
    def order_number_format(cls, v):
        for t in v:
            if not ORDER_NUMBER_PATTERN.fullmatch(t):
                raise ValueError(f"'{t}' is not a valid order number; "
                                 "phases without tickets must use an empty list")
        return v


class StorySummary(BaseModel):
    initial_issue: Section
    follow_ups: Section
    developments: Section
    later_incidents: Section
    recent_events: Section


SECTION_TITLES = [
    ("initial_issue", "Initial Issue"),
    ("follow_ups", "Follow-ups"),
    ("developments", "Developments"),
    ("later_incidents", "Later Incidents"),
    ("recent_events", "Recent Events"),
]

# the schema-level rejection in action: N/A is not data
try:
    Section(ticket_numbers=["N/A"], narrative="x")
except Exception as e:
    print("rejected:", str(e).splitlines()[2].strip())

rejected: Value error, 'N/A' is not a valid order number; phases without tickets must use an empty list [type=value_error, input_value=['N/A'], input_type=list]


### Explore: one structured call

`with_structured_output(StorySummary, method="json_schema")` makes the API
enforce the shape during generation - the return value is not text but an
already-validated `StorySummary` object.

In [35]:
structured_llm = llm.with_structured_output(StorySummary, method="json_schema")

group = df_products[(df_products["CUSTOMER_NUMBER"] == "123") & (df_products["PRODUCT"] == "GIGA")]
demo_story = structured_llm.invoke(STORY_PROMPT.format(
    customer="123", product="GIGA",
    categories="GIGA", tickets=format_tickets(group)))
demo_story

StorySummary(initial_issue=Section(ticket_numbers=['001-0670932/24'], narrative='The customer experienced a failure when attempting to activate the Gigabit Option.'), follow_ups=Section(ticket_numbers=[], narrative='No further tickets recorded in this period.'), developments=Section(ticket_numbers=[], narrative='No further tickets recorded in this period.'), later_incidents=Section(ticket_numbers=[], narrative='No further tickets recorded in this period.'), recent_events=Section(ticket_numbers=[], narrative='No further tickets recorded in this period.'))

### Explore: the evaluation - two checklists against the data

`cited - actual` = tickets the model made up. `actual - cited` = real tickets
the story ignored. Both empty = the references are proven.

In [36]:
def evaluate_story(story: StorySummary, group: pd.DataFrame) -> list:
    """Empty list = every reference is real and complete."""
    cited = set()
    for field, _ in SECTION_TITLES:
        cited |= set(getattr(story, field).ticket_numbers)
    actual = set(group["ORDER_NUMBER"])

    problems = []
    if cited - actual:
        problems.append(f"cited tickets that do not exist in this data: {sorted(cited - actual)}")
    if actual - cited:
        problems.append(f"never mentioned these real tickets: {sorted(actual - cited)}")
    return problems

print("real output:", evaluate_story(demo_story, group))

broken = demo_story.model_copy(deep=True)
broken.initial_issue.ticket_numbers = ["001-0679999/24"]   # a fake number, valid format
print("broken copy:", evaluate_story(broken, group))

real output: []
broken copy: ["cited tickets that do not exist in this data: ['001-0679999/24']", "never mentioned these real tickets: ['001-0670932/24']"]


### Explore: timeframes computed, rendering owned by us

The timeframe comes from the verified tickets' real ACCEPTANCE_TIMEs - pandas
writes the date line, the model never does. Empty phases get one uniform
wording, decided here, not improvised per summary.

In [37]:
def section_timeframe(section: Section, group: pd.DataFrame) -> str:
    if not section.ticket_numbers:
        return "No tickets recorded in this period"
    dates = group.loc[group["ORDER_NUMBER"].isin(section.ticket_numbers), "ACCEPTANCE_TIME"]
    if dates.min().date() == dates.max().date():
        return f"{dates.min():%b %d, %Y}"
    return f"{dates.min():%b %d} - {dates.max():%b %d, %Y}"


def render_story(story: StorySummary, group: pd.DataFrame) -> str:
    parts = []
    for i, (field, title) in enumerate(SECTION_TITLES, 1):
        s = getattr(story, field)
        tickets = ", ".join(s.ticket_numbers) if s.ticket_numbers else "None"
        parts.append(f"### {i}. **{title}**\n"
                     f"- **Timeframe:** {section_timeframe(s, group)}\n"
                     f"- **Ticket Numbers:** {tickets}\n"
                     f"- **Narrative:** {s.narrative}")
    return "\n\n".join(parts)

from IPython.display import Markdown, display
display(Markdown(render_story(demo_story, group)))

### 1. **Initial Issue**
- **Timeframe:** Nov 15, 2024
- **Ticket Numbers:** 001-0670932/24
- **Narrative:** The customer experienced a failure when attempting to activate the Gigabit Option.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### The function: generate -> evaluate -> corrective retry

All the pieces assembled. The correction message names the exact mistakes and
hands over the full real ticket list - the model fixes its own work; our code
never edits an answer. Status travels with every summary:
`verified` (clean first try), `corrected` (clean after retry),
`unverified` (retries exhausted - shown with a warning, never silently).

In [38]:
def summarize_group(group, structured_llm, customer, product, max_retries=2):
    base_prompt = STORY_PROMPT.format(
        customer=customer, product=product,
        categories=", ".join(sorted(group["SERVICE_CATEGORY"].unique())),
        tickets=format_tickets(group))
    prompt, story = base_prompt, None
    for attempt in range(max_retries + 1):
        try:
            story = structured_llm.invoke(prompt)
            problems = evaluate_story(story, group)
        except Exception as err:
            problems = [f"the output did not match the required schema: {err}"]
        if story is not None and not problems:
            return story, ("verified" if attempt == 0 else "corrected")
        actual = ", ".join(sorted(group["ORDER_NUMBER"]))
        prompt = (base_prompt
                  + "\n\nCorrection needed - your previous answer had these problems:\n"
                  + "\n".join(f"- {p}" for p in problems)
                  + f"\nRegenerate, citing exactly these tickets across the phases: {actual}.")
    if story is None:
        raise RuntimeError(f"no valid output after {max_retries + 1} attempts")
    return story, "unverified"

story, status = summarize_group(group, structured_llm, "123", "GIGA")
print("status:", status)

status: verified


In [39]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ORDER_NUMBER                 32 non-null     str           
 1   ORDER_ID                     32 non-null     str           
 2   ACCEPTANCE_TIME              32 non-null     datetime64[us]
 3   COMPLETION_TIME              32 non-null     datetime64[us]
 4   CUSTOMER_NUMBER              32 non-null     str           
 5   CUSTOMER_COUNT               32 non-null     str           
 6   ORDER_TYPE                   32 non-null     str           
 7   ORDER_CLASS                  32 non-null     str           
 8   PROCESSING_STATUS            32 non-null     str           
 9   SERVICE_CATEGORY             32 non-null     str           
 10  ORDER_DESCRIPTION_1          32 non-null     str           
 11  ORDER_DESCRIPTION_2          32 non-null     str          

## Step 9: End-to-end LangGraph pipeline

Every step above becomes a **graph node**, not just the LLM call:

`load -> repair -> convert -> clean -> filter -> map_products -> summarize -> END`

The graph state carries the data between nodes: `load` puts the parsed `header` + `rows` (plain lists) into the state, `repair` fixes the shifted rows and builds the DataFrame, and every node after that transforms `df`. Each node is a thin wrapper around a function defined earlier - the graph adds orchestration, not new logic. `repair` runs before `convert`, so the saved CSV/Excel already contain the corrected columns.

In [ ]:
import time

from typing import Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class PipelineState(TypedDict, total=False):
    raw_path: str          # input: path to the raw ticket .txt
    header: list           # column names from the raw file
    rows: list             # ticket rows as plain lists (repaired in place)
    df: Optional[pd.DataFrame]   # working DataFrame, transformed node by node
    summaries: dict        # product -> storytelling summary text


def node_load(state: PipelineState) -> PipelineState:
    header, rows = load_raw_tickets(state["raw_path"])
    return {"header": header, "rows": rows}

def node_repair(state: PipelineState) -> PipelineState:
    rows = repair_shifted_rows(state["rows"], state["header"])
    return {"df": pd.DataFrame(rows, columns=state["header"])}

def node_convert(state: PipelineState) -> PipelineState:
    state["df"].to_csv(DATA_DIR / "tickets_converted.csv", index=False)
    state["df"].to_excel(DATA_DIR / "tickets_converted.xlsx", index=False)
    return {}

def node_clean(state: PipelineState) -> PipelineState:
    return {"df": clean_tickets(state["df"])}

def node_filter(state: PipelineState) -> PipelineState:
    return {"df": filter_categories(state["df"])}

def node_map(state: PipelineState) -> PipelineState:
    return {"df": map_products(state["df"])}

def node_summarize(state: PipelineState) -> PipelineState:
    structured = llm.with_structured_output(StorySummary, method="json_schema")
    summaries = {}
    for (cust, product), group in state["df"].groupby(["CUSTOMER_NUMBER", "PRODUCT"]):
        print(f"Summarizing customer {cust} - {product} ({len(group)} tickets) ...")
        t0 = time.time()
        story, status = summarize_group(group, structured, cust, product)
        print(f"  -> {status} in {time.time() - t0:.1f}s")
        summaries.setdefault(cust, {})[product] = {
            "markdown": render_story(story, group),
            "status": status,
        }
    return {"summaries": summaries}


builder = StateGraph(PipelineState)

builder.add_node("load", node_load)
builder.add_node("repair", node_repair)
builder.add_node("convert", node_convert)
builder.add_node("clean", node_clean)
builder.add_node("filter", node_filter)
builder.add_node("map_products", node_map)
builder.add_node("summarize", node_summarize)

builder.add_edge(START, "load")
builder.add_edge("load", "repair")
builder.add_edge("repair", "convert")
builder.add_edge("convert", "clean")
builder.add_edge("clean", "filter")
builder.add_edge("filter", "map_products")
builder.add_edge("map_products", "summarize")
builder.add_edge("summarize", END)

pipeline = builder.compile()
pipeline

In [41]:
summaries = {}
for (cust, product), group in df_products.groupby(["CUSTOMER_NUMBER", "PRODUCT"]):
    cats = ", ".join(sorted(group["SERVICE_CATEGORY"].unique()))
    print(f"Summarizing customer {cust} - {product} ({len(group)} tickets) ...")


Summarizing customer 123 - Broadband (2 tickets) ...
Summarizing customer 123 - GIGA (1 tickets) ...
Summarizing customer 123 - Hardware (5 tickets) ...
Summarizing customer 123 - TV (1 tickets) ...
Summarizing customer 123 - VOD (1 tickets) ...
Summarizing customer 123 - Voice (1 tickets) ...
Summarizing customer 456 - Broadband (2 tickets) ...
Summarizing customer 456 - GIGA (1 tickets) ...
Summarizing customer 456 - Hardware (4 tickets) ...
Summarizing customer 456 - TV (1 tickets) ...
Summarizing customer 456 - VOD (1 tickets) ...
Summarizing customer 456 - Voice (1 tickets) ...
Summarizing customer 789 - Broadband (2 tickets) ...
Summarizing customer 789 - GIGA (1 tickets) ...
Summarizing customer 789 - Hardware (5 tickets) ...
Summarizing customer 789 - TV (1 tickets) ...
Summarizing customer 789 - VOD (1 tickets) ...
Summarizing customer 789 - Voice (1 tickets) ...


## Step 10: Run the graph
One `invoke` executes the whole pipeline: raw text file in -> storytelling summaries out.

In [42]:
result = pipeline.invoke({"raw_path": str(RAW_FILE)})

from IPython.display import Markdown, display
BADGE = {"verified": "verified (references proven first try)",
         "corrected": "corrected (fixed after one retry)",
         "unverified": "UNVERIFIED - read with care"}
for cust, products in result["summaries"].items():
    display(Markdown(f"---\n# Customer {cust}"))
    for product, entry in products.items():
        display(Markdown(f"## {product} - {BADGE[entry['status']]}\n\n{entry['markdown']}"))

Loaded 42 tickets x 38 columns - all rows complete
Repaired 19 shifted tickets: ['001-0682299/24', '001-0670955/24', '001-0671372/24', '001-0670932/24', '001-0670744/24', '001-0671183/24', '001-0670956/24', '001-0671373/24', '001-0670933/24', '001-0670745/24', '001-0671184/24', '001-0682301/24', '001-0671185/24', '001-0682302/24', '001-0670957/24', '001-0671374/24', '001-0670934/24', '001-0670746/24', '001-0671186/24']
Dropped 10 all-empty columns: ['ORDER_UNIT_ID', 'ADDITIONAL_ORDER_DESCRIPTION_MAXIMUM', 'SUBUNIT_NAME', 'ASSIGNMENT_TIME_MINIMUM', 'IMIL_TIME_MINIMUM', 'START_TIME_MINIMUM', 'ASSIGNMENT_TIME', 'ASSIGNED_BY_NAME', 'ASSIGNMENT_PROCESSING_STATUS', 'ASSIGNMENT_ADDITIONAL_INFO']
Category counts BEFORE filtering:
SERVICE_CATEGORY
HDW     14
SOP      6
NTF      4
NET      3
KAI      3
KAV      3
GIGA     3
VOD      3
KAD      3

Kept 32 / 42 tickets
Summarizing customer 123 - Broadband (2 tickets) ...
Summarizing customer 123 - GIGA (1 tickets) ...
Summarizing customer 123 - Ha

---
# Customer 123

## Broadband - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 13, 2024
- **Ticket Numbers:** 001-0682299/24
- **Narrative:** The customer reported slow internet speeds, and the bandwidth was increased to resolve the issue.

### 2. **Follow-ups**
- **Timeframe:** Nov 14, 2024
- **Ticket Numbers:** 001-0670955/24
- **Narrative:** The customer reported a lack of reception due to a defective CI module, which was then ordered for replacement.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## GIGA - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 15, 2024
- **Ticket Numbers:** 001-0670932/24
- **Narrative:** The customer experienced a failure when attempting to activate the Gigabit Option.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## Hardware - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 05, 2024
- **Ticket Numbers:** 001-0671177/24
- **Narrative:** The customer first reported slow WLAN speeds on November 5, 2024, which were resolved by optimizing the WLAN settings.

### 2. **Follow-ups**
- **Timeframe:** Nov 06, 2024
- **Ticket Numbers:** 001-0670952/24, 001-0671369/24
- **Narrative:** On November 6, 2024, the customer experienced a total loss of internet connection and unstable WLAN, which were addressed by restarting the router and changing the channel.

### 3. **Developments**
- **Timeframe:** Nov 07, 2024
- **Ticket Numbers:** 001-0671178/24
- **Narrative:** The issue persisted on November 7, 2024, with a complete loss of WLAN connection, requiring a router reset.

### 4. **Later Incidents**
- **Timeframe:** Nov 13, 2024
- **Ticket Numbers:** 001-0671182/24
- **Narrative:** On November 13, 2024, the customer reported that the router would not start, leading to the shipment of a replacement device.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

...

## GIGA - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 19, 2024
- **Ticket Numbers:** 001-0670934/24
- **Narrative:** The customer reported a delay in the activation of the Gigabit Option and noted that the confirmation was missing.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## Hardware - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 10, 2024
- **Ticket Numbers:** 001-0671180/24
- **Narrative:** The customer experienced WLAN stability issues on November 10, 2024, which were temporarily resolved by restarting the router.

### 2. **Follow-ups**
- **Timeframe:** Nov 11, 2024
- **Ticket Numbers:** 001-0670954/24, 001-0671371/24
- **Narrative:** On November 11, 2024, the customer reported a total loss of internet connection and slow WLAN speeds, leading to troubleshooting and a change in the frequency band.

### 3. **Developments**
- **Timeframe:** Nov 12, 2024
- **Ticket Numbers:** 001-0671181/24
- **Narrative:** By November 12, 2024, the WLAN connection was completely lost, necessitating the order of a technician.

### 4. **Later Incidents**
- **Timeframe:** Nov 16, 2024
- **Ticket Numbers:** 001-0671185/24
- **Narrative:** On November 16, 2024, the router was identified as defective, and a technician was commissioned for repair or replacement.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## TV - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 20, 2024
- **Ticket Numbers:** 001-0671186/24
- **Narrative:** The customer reported an issue where the invoice was not displayed in the customer portal, which was identified as a technical problem and resolved.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## VOD - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 19, 2024
- **Ticket Numbers:** 001-0670746/24
- **Narrative:** The customer experienced a playback failure with Video on Demand, resulting in an error message, which was resolved via TITAN.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## Voice - verified (references proven first try)

### 1. **Initial Issue**
- **Timeframe:** Nov 18, 2024
- **Ticket Numbers:** 001-0671374/24
- **Narrative:** The customer reported picture interference and artifacts on the HD channel.

### 2. **Follow-ups**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 3. **Developments**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets tickets recorded in this period.

### 4. **Later Incidents**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

### 5. **Recent Events**
- **Timeframe:** No tickets recorded in this period
- **Ticket Numbers:** None
- **Narrative:** No further tickets recorded in this period.

## Step 11 (Bonus): business insights

Same analysis as the Streamlit Insights tab, cell by cell. Three questions
every support dataset must answer:

1. **What breaks the most?** (where the volume comes from)
2. **Do fixes hold?** (repeat contacts = failed first fixes)
3. **How fast and how costly is handling?**

Each chart ends with a computed finding, not a hardcoded one.

In [43]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

# German device names -> English so one issue is not split into two bars by language
GERMAN_TERMS = {"Kabelrouter": "Cable Router", "CI-Modul": "CI Module",
                "Sender": "TV Channel", "Kundenportal": "Customer Portal",
                "Filme": "Movies"}
d = df_products.assign(ISSUE=df_products["ORDER_DESCRIPTION_1"].fillna("?").replace(GERMAN_TERMS))
d["ISSUE"].value_counts()

ISSUE
Cable Router       14
Internet            3
Gigabit Option      3
Smartcard           2
HD Channel          2
Video on Demand     2
Customer Portal     2
CI Module           1
TV Channel          1
Movies              1
Password            1
Name: count, dtype: int64

### Q1. What breaks the most?

How to read the Pareto: bars = tickets per device/service, sorted biggest first;
the line = cumulative share of total volume. Where the line crosses ~80% you have
found the few issues that cause most of the volume (the 80/20 rule). The point of
failure is whatever the first bar is.

In [44]:
issue_counts = d["ISSUE"].value_counts()
cum = issue_counts.cumsum() / issue_counts.sum() * 100

pareto = go.Figure()
pareto.add_bar(x=issue_counts.index, y=issue_counts.values, name="tickets")
pareto.add_scatter(x=issue_counts.index, y=cum.values, name="cumulative %",
                   yaxis="y2", mode="lines+markers")
pareto.update_layout(title="Pareto: ticket volume per device/service",
                     yaxis2=dict(overlaying="y", side="right", range=[0, 110]))
pareto.show()

print(f"Finding: {issue_counts.iloc[0] / len(d):.0%} of tickets "
      f"({int(issue_counts.iloc[0])} of {len(d)}) are one device: '{issue_counts.index[0]}'")

Finding: 44% of tickets (14 of 32) are one device: 'Cable Router'


In [45]:
# sorted horizontal bar, not a pie: 6 products are too many slices to compare by angle
prod_counts = d["PRODUCT"].value_counts().sort_values().reset_index()
prod_counts.columns = ["product", "tickets"]
px.bar(prod_counts, x="tickets", y="product", orientation="h",
       title="Tickets per product").show()

### Q2. Do fixes hold? (repeat contacts)

Grouped by customer + PRODUCT (not category, so a NET ticket followed by a KAI
ticket counts as a Broadband repeat). More than one contact in a group means the
previous fix did not hold. The "final resolution" column shows how each chain
eventually ended - in this data, every chain ends in a replacement or technician,
so the intermediate remote fixes bought nothing.

In [46]:
chains = []
for (cust, prod), g in d.sort_values("ACCEPTANCE_TIME").groupby(["CUSTOMER_NUMBER", "PRODUCT"]):
    if len(g) > 1:
        days = g["ACCEPTANCE_TIME"].diff().dt.total_seconds().div(86400).dropna()
        chains.append({
            "customer": cust,
            "product": prod,
            "categories": ", ".join(sorted(g["SERVICE_CATEGORY"].unique())),
            "contacts": len(g),
            "avg days between contacts": round(days.mean(), 1),
            "final resolution": g["COMPLETION_RESULT_KB"].iloc[-1],
        })

repeat_df = pd.DataFrame(chains).sort_values("contacts", ascending=False)
repeat_tickets = int((repeat_df["contacts"] - 1).sum())
print(f"Finding: {repeat_tickets} repeat tickets = {repeat_tickets / len(d):.0%} of the volume")
repeat_df

Finding: 14 repeat tickets = 44% of the volume


,customer,product,categories,contacts,avg days between contacts,final resolution
1,123,Hardware,HDW,5,2.0,Austauschgerät versendet
5,789,Hardware,HDW,5,1.5,Technician Commissioned
3,456,Hardware,HDW,4,1.3,New Router Sent
0,123,Broadband,"KAI, NET",2,0.8,Neues CI-Modul bestellt
2,456,Broadband,"KAI, NET",2,1.1,Smartcard Replaced
4,789,Broadband,"KAI, NET",2,0.8,PIN Forgotten


### Q3. How fast and how costly is handling?

Left: the resolution-time distribution per product (box = middle 50%, line =
median). Right: how each product's tickets were closed - cheap remote fix vs the
expensive path (technician visit or hardware shipment), classified from keywords
in COMPLETION_RESULT_KB.

In [47]:
ESCALATION_WORDS = ("technician", "techniker", "replaced", "sent", "ordered",
                    "commissioned", "forwarded", "weitergeleitet", "versendet", "bestellt")
handled = d["COMPLETION_RESULT_KB"].fillna("").str.lower()
d["HANDLING"] = handled.apply(
    lambda t: "Escalated / hardware" if any(w in t for w in ESCALATION_WORDS) else "Remote fix")

px.box(d, x="PRODUCT", y="RESOLUTION_MINUTES",
       title="Resolution time distribution (min)").show()
px.histogram(d, x="PRODUCT", color="HANDLING", barmode="stack",
             title="Remote fixes vs escalations per product").show()

esc_share = (d["HANDLING"] != "Remote fix").mean()
print(f"Finding: resolution takes {d['RESOLUTION_MINUTES'].min():.0f}-"
      f"{d['RESOLUTION_MINUTES'].max():.0f} min (speed is fine); "
      f"{esc_share:.0%} of tickets take the expensive escalation path")

Finding: resolution takes 15-60 min (speed is fine); 28% of tickets take the expensive escalation path


### More views - need more data to be conclusive

With ~2 weeks of tickets these are illustrative only; at real volume they answer
staffing (when does load hit) and account health (who keeps calling).

In [48]:
daily = d.groupby([d["ACCEPTANCE_TIME"].dt.date, "PRODUCT"]).size().reset_index(name="tickets")
daily.columns = ["date", "product", "tickets"]
px.bar(daily, x="date", y="tickets", color="product",
       title="Daily ticket volume by product").show()

In [49]:
tmp = d.assign(weekday=d["ACCEPTANCE_TIME"].dt.day_name(), hour=d["ACCEPTANCE_TIME"].dt.hour)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (tmp.pivot_table(index="weekday", columns="hour", values="ORDER_NUMBER", aggfunc="count")
           .reindex([x for x in order if x in tmp["weekday"].unique()]))
px.imshow(heat, labels=dict(x="hour of day", y="", color="tickets"),
          title="When tickets arrive (weekday x hour)").show()

In [50]:
px.scatter(d, x="ACCEPTANCE_TIME", y="CUSTOMER_NUMBER", color="PRODUCT",
           hover_data=["ORDER_NUMBER", "NOTE_MAXIMUM"],
           title="Customer ticket timeline").show()

## Wrap-up

- The full pipeline (load -> convert -> clean -> filter -> map -> summarize) runs as **one LangGraph graph**.
- Key business signals: WLAN/router (HDW) dominates volume; recurring no-connection issues; all tickets resolve within 15-60 min.
- **Next step:** the same pipeline is packaged into `pipeline.py` and served by the Streamlit app in `app.py` (upload a raw .txt -> get per-product storytelling summaries + charts).

### Scratch: raw tickets per customer (before filtering)